# =============================================================================
# NYC Yellow Taxi Trip Data Analysis with PySpark
# ============================================
# Dataset : NYC TLC Yellow Taxi Trip Records (Jan–Feb 2026)
# Source   : https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Tools    : PySpark, Google Colab
# Author   : Amna
# =============================================================================
#
# This notebook demonstrates core PySpark skills applied to a real-world
# ride-hailing dataset, covering:
#   - SparkSession setup and Parquet ingestion
#   - Schema inspection and descriptive statistics
#   - Filtering, selecting, and sorting DataFrames
#   - Null handling and data cleaning
#   - Feature engineering (trip duration)
#   - Unioning multiple monthly files
#   - Joining with a lookup table
#   - GroupBy aggregations
# =============================================================================


In [2]:
# ── 0. ENVIRONMENT SETUP ─────────────────────────────────────────────────────

In [3]:
# Install PySpark in the Colab environment
# (Skip if running locally with PySpark already installed)
!pip install pyspark --quiet

In [4]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.0.2


In [5]:
# Mount Google Drive (where the dataset files live)
from google.colab import drive
drive.mount('/content/drive/')

import os

# Quick sanity-check: confirm the lookup CSV is accessible
lookup_path = '/content/drive/MyDrive/PySpark-Project-Dataset/taxi_zone_lookup.csv'
print(f"Lookup file exists: {os.path.isfile(lookup_path)}")

Mounted at /content/drive/
Lookup file exists: True


In [6]:
# ── 1. SPARK SESSION ─────────────────────────────────────────────────────────

from pyspark.sql import SparkSession

# getOrCreate() reuses an existing session if one is already running —
# safe to call multiple times without spinning up duplicate clusters.
spark = SparkSession.builder \
    .appName("NYC Taxi Analysis") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.0.2


In [7]:
# ── 2. LOAD DATA ─────────────────────────────────────────────────────────────

# Read the January 2026 Parquet file into a Spark DataFrame.
# Parquet is a columnar format — much faster than CSV for large datasets.
jan_path = '/content/drive/MyDrive/PySpark-Project-Dataset/yellow_tripdata_2026-01.parquet'
df = spark.read.parquet(jan_path)

# Preview the first 5 rows
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [8]:
# ── 3. SCHEMA & BASIC EXPLORATION ────────────────────────────────────────────

# Total number of rows
print(f"Row count: {df.count():,}")

# Column names as a Python list
print("Columns:", df.schema.names)

# Full schema with data types — useful for spotting type mismatches early
df.printSchema()

# Descriptive statistics for numeric columns of interest
df.describe(['passenger_count', 'total_amount']).show()


Row count: 3,724,889
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (

In [9]:
# ── 4. BASIC SELECTION & SORTING ────────────────────────────────────────────

In [10]:
 # Select a subset of columns
df.select('passenger_count', 'total_amount').show()

# Sort by total_amount descending to spot high-value or anomalous trips
df.sort('total_amount', ascending=False).show()

+---------------+------------+
|passenger_count|total_amount|
+---------------+------------+
|              1|       15.86|
|              0|       13.65|
|              0|       18.95|
|              4|       55.56|
|              0|        23.1|
|              2|       24.94|
|              1|       17.15|
|              0|        34.0|
|              1|       51.66|
|              3|       18.06|
|              1|       42.42|
|              1|       21.42|
|              2|       27.65|
|              3|       10.15|
|              1|         0.0|
|              1|       22.75|
|              1|       41.58|
|              1|       41.55|
|              1|        63.4|
|              1|       18.04|
+---------------+------------+
only showing top 20 rows
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+-------

In [11]:
# ── 5. FILTERING ─────────────────────────────────────────────────────────────

In [12]:
# Filter for airport trips only (Airport_fee > 0 flags JFK / LaGuardia rides)
df.filter('Airport_fee > 0').show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:08:06|  2026-01-01 00:33:41|              1|        10.48|         1|                 N|         138|    

In [13]:
# Compound filter: airport trips with more than 2 passengers
df.filter('Airport_fee > 0 AND passenger_count > 2').show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:08:07|  2026-01-01 00:40:39|              3|        17.18|         2|                 N|         132|    

In [14]:
# Same filter — but project only the columns we care about
df.filter('Airport_fee > 0 AND passenger_count > 2') \
  .select('VendorID', 'passenger_count', 'total_amount') \
  .show()

+--------+---------------+------------+
|VendorID|passenger_count|total_amount|
+--------+---------------+------------+
|       2|              3|       80.75|
|       2|              4|       71.64|
|       2|              3|       82.27|
|       2|              4|        76.5|
|       2|              3|       26.15|
|       2|              3|       68.35|
|       2|              4|       60.45|
|       2|              4|      111.79|
|       2|              3|      156.15|
|       2|              4|       26.83|
|       2|              4|       48.55|
|       2|              3|       85.19|
|       2|              4|       95.88|
|       2|              3|        33.9|
|       2|              4|       23.35|
|       2|              3|       36.91|
|       2|              4|       26.83|
|       2|              3|        7.25|
|       2|              3|       92.69|
|       2|              4|       53.45|
+--------+---------------+------------+
only showing top 20 rows


In [15]:
# ── Challenge: Non-airport solo rides, ordered by trip distance ──
# Combines filter + select + sort in a single readable chain.
# Useful for understanding typical solo city ride patterns.
df.filter('passenger_count = 1 AND Airport_fee = 0') \
  .select('trip_distance', 'total_amount') \
  .sort('trip_distance', ascending=False) \
  .show()

+-------------+------------+
|trip_distance|total_amount|
+-------------+------------+
|     98773.58|         8.6|
|       295.99|       19.74|
|       257.24|      498.46|
|        235.5|      576.75|
|        223.6|        26.0|
|        192.8|        39.6|
|       173.72|       390.8|
|        157.3|        27.0|
|        151.6|        37.0|
|       142.92|      -670.4|
|       142.92|       670.4|
|       121.18|       838.9|
|       116.46|      -827.7|
|       116.46|       827.7|
|       113.99|       501.0|
|        101.9|        20.0|
|       101.65|     -680.61|
|       101.65|      680.61|
|       100.37|       21.06|
|        99.93|      652.21|
+-------------+------------+
only showing top 20 rows


In [16]:
# ── 6. NULL HANDLING ─────────────────────────────────────────────────────────

In [17]:
from pyspark.sql.functions import col, isnull

# Count nulls in key columns — important before any aggregation or modelling
null_fare     = df.filter(isnull(col('fare_amount'))).count()
null_pax      = df.filter(isnull(col('passenger_count'))).count()
print(f"Null fare_amount rows   : {null_fare:,}")
print(f"Null passenger_count rows: {null_pax:,}")

# Fill missing passenger counts with 1 (reasonable default for solo rides)
df_clean = df.fillna({'passenger_count': 1})

# Verify no nulls remain in that column
remaining_nulls = df_clean.filter(isnull(col('passenger_count'))).count()
print(f"Null passenger_count after fill: {remaining_nulls}")


Null fare_amount rows   : 0
Null passenger_count rows: 1,088,058
Null passenger_count after fill: 0


In [18]:
# ── 7. FEATURE ENGINEERING — TRIP DURATION ───────────────────────────────────

In [19]:
from pyspark.sql.functions import unix_timestamp, round as spark_round

# Compute trip duration in minutes from pickup/dropoff timestamps.
# unix_timestamp() converts datetime strings to epoch seconds;
# dividing the difference by 60 gives minutes.
df_featured = df_clean.withColumn(
    'trip_duration_minutes',
    spark_round(
        (unix_timestamp('tpep_dropoff_datetime') - unix_timestamp('tpep_pickup_datetime')) / 60,
        1   # Round to 1 decimal place
    )
)

df_featured.select(
    'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_duration_minutes'
).show(10)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
| 2026-01-01 00:54:04|  2026-01-01 00:59:37|                  5.6|
| 2026-01-01 00:34:04|  2026-01-01 00:39:47|                  5.7|
| 2026-01-01 00:57:06|  2026-01-01 01:05:59|                  8.9|
| 2026-01-01 00:15:22|  2026-01-01 00:58:10|                 42.8|
| 2026-01-01 00:27:13|  2026-01-01 00:40:43|                 13.5|
| 2026-01-01 00:47:11|  2026-01-01 01:00:47|                 13.6|
| 2026-01-01 00:17:54|  2026-01-01 00:28:32|                 10.6|
| 2026-01-01 00:34:28|  2026-01-01 00:59:05|                 24.6|
| 2026-01-01 00:34:14|  2026-01-01 01:11:58|                 37.7|
| 2026-01-01 00:41:07|  2026-01-01 00:50:42|                  9.6|
+--------------------+---------------------+---------------------+
only showing top 10 rows


In [20]:
# ── 8. COLUMN SELECTION & RENAMING ───────────────────────────────────────────

In [21]:
# Keep only the columns needed for downstream analysis and apply
# consistent snake_case naming conventions.
df_renamed = df_featured.select(
    'VendorID',
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'total_amount',
    'passenger_count',
    'trip_distance',
    'trip_duration_minutes'
).withColumnsRenamed({
    'VendorID'             : 'vendor_id',
    'tpep_pickup_datetime' : 'pu_datetime',
    'tpep_dropoff_datetime': 'do_datetime',
    'PULocationID'         : 'pu_location_id',
    'DOLocationID'         : 'do_location_id'
})

df_renamed.show(5)

+---------+-------------------+-------------------+--------------+--------------+------------+---------------+-------------+---------------------+
|vendor_id|        pu_datetime|        do_datetime|pu_location_id|do_location_id|total_amount|passenger_count|trip_distance|trip_duration_minutes|
+---------+-------------------+-------------------+--------------+--------------+------------+---------------+-------------+---------------------+
|        2|2026-01-01 00:54:04|2026-01-01 00:59:37|           239|           238|       15.86|              1|         0.97|                  5.6|
|        1|2026-01-01 00:34:04|2026-01-01 00:39:47|           163|           162|       13.65|              0|          0.9|                  5.7|
|        1|2026-01-01 00:57:06|2026-01-01 01:05:59|            43|           237|       18.95|              0|          1.4|                  8.9|
|        2|2026-01-01 00:15:22|2026-01-01 00:58:10|           142|           209|       55.56|              4|        

In [22]:
# ── 9. COMBINING MONTHLY FILES ────────────────────────────────────────────────

In [23]:
# Read January and February 2026 files separately, then union them.
# union() requires both DataFrames to have identical schemas.
feb_path = '/content/drive/MyDrive/PySpark-Project-Dataset/yellow_tripdata_2026-02.parquet'

df_jan = spark.read.parquet(jan_path)
df_feb = spark.read.parquet(feb_path)

df_2026 = df_jan.union(df_feb)

print(f"Jan rows : {df_jan.count():,}")
print(f"Feb rows : {df_feb.count():,}")
print(f"Combined : {df_2026.count():,}")

Jan rows : 3,724,889
Feb rows : 3,399,866
Combined : 7,124,755


In [24]:
# ── 10. ZONE LOOKUP JOIN ──────────────────────────────────────────────────────

In [25]:
# Load the taxi zone reference table (Borough, Zone, service_zone per LocationID)
zone_lookup = spark.read \
    .option('header', 'true') \
    .csv(lookup_path)

zone_lookup.show(5)

# Spot-check: confirm zone data for a specific LocationID
zone_lookup.filter('LocationID = 43').show()

# Left join: enrich trip data with pickup zone information.
# 'left' ensures we keep all trips even if a LocationID has no lookup match.
df_joined = df_2026.join(
    zone_lookup,
    df_2026.PULocationID == zone_lookup.LocationID,
    how='left'
)

df_joined.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
+----------+---------+------------+------------+
|LocationID|  Borough|        Zone|service_zone|
+----------+---------+------------+------------+
|        43|Manhattan|Central Park| Yellow Zone|
+----------+---------+------------+------------+

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+---

In [26]:
# ── 11. FULL PIPELINE — 2025 CHALLENGE SOLUTION ───────────────────────────────
#
# This section demonstrates a complete, production-style pipeline:
# read → union → select/rename → join → clean up columns
# ─────────────────────────────────────────────────────────────────────────────

In [29]:
# Step 1 — Read monthly Parquet files
df_jan_2025 = spark.read.parquet(
    '/content/drive/MyDrive/PySpark-Project-Dataset/yellow_tripdata_2026-01.parquet')
df_feb_2025 = spark.read.parquet(
    '/content/drive/MyDrive/PySpark-Project-Dataset/yellow_tripdata_2026-02.parquet')

In [30]:
# Step 2 — Combine into a single DataFrame
df_2025 = df_jan_2025.union(df_feb_2025)

# Step 3 — Project relevant columns and standardise names
df_2025 = df_2025.select(
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'fare_amount',
    'Airport_fee',
    'total_amount',
    'payment_type'
).withColumnsRenamed({
    'tpep_pickup_datetime' : 'pu_datetime',
    'tpep_dropoff_datetime': 'do_datetime',
    'PULocationID'         : 'pu_location_id',
    'DOLocationID'         : 'do_location_id',
    'Airport_fee'          : 'airport_fee'
})

In [32]:
# Step 4 — Load the zone lookup table
taxi_zones = spark.read \
    .option('header', 'true') \
    .csv('/content/drive/MyDrive/PySpark-Project-Dataset/taxi_zone_lookup.csv')

# Step 5 — Join on dropoff location to get the dropoff borough
df_2025 = df_2025.join(
    taxi_zones,
    df_2025.do_location_id == taxi_zones.LocationID,
    how='left'
)

# Step 6 — Drop redundant lookup columns; rename Borough for clarity
df_2025 = df_2025 \
    .drop('LocationID', 'Zone', 'service_zone') \
    .withColumnsRenamed({'Borough': 'do_boro'})

# Step 7 — Final result
df_2025.show(10)
print(f"Final 2025 dataset rows: {df_2025.count():,}")

+-------------------+-------------------+--------------+--------------+---------------+-----------+-----------+------------+------------+---------+
|        pu_datetime|        do_datetime|pu_location_id|do_location_id|passenger_count|fare_amount|airport_fee|total_amount|payment_type|  do_boro|
+-------------------+-------------------+--------------+--------------+---------------+-----------+-----------+------------+------------+---------+
|2026-01-01 00:54:04|2026-01-01 00:59:37|           239|           238|              1|        7.2|        0.0|       15.86|           1|Manhattan|
|2026-01-01 00:34:04|2026-01-01 00:39:47|           163|           162|              0|        7.9|        0.0|       13.65|           2|Manhattan|
|2026-01-01 00:57:06|2026-01-01 01:05:59|            43|           237|              0|       10.7|        0.0|       18.95|           1|Manhattan|
|2026-01-01 00:15:22|2026-01-01 00:58:10|           142|           209|              4|       38.7|        0.0| 

In [33]:
# ── 12. AGGREGATIONS ─────────────────────────────────────────────────────────

In [34]:
from pyspark.sql.functions import avg, count, max as spark_max, min as spark_min

# Payment type breakdown — how do passengers prefer to pay?
# Payment codes: 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown
df.groupBy('payment_type') \
  .count() \
  .sort('payment_type') \
  .show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0|1088058|
|           1|2249747|
|           2| 314043|
|           3|  16641|
|           4|  56400|
+------------+-------+



In [35]:
# Average fare by payment type — do credit card users take longer/pricier trips?
df.groupBy('payment_type') \
  .agg(avg('total_amount').alias('avg_total_amount')) \
  .sort('payment_type') \
  .show()

+------------+------------------+
|payment_type|  avg_total_amount|
+------------+------------------+
|           0| 31.68379475177417|
|           1| 29.61055066192971|
|           2|23.411882226317573|
|           3| 8.927757947238725|
|           4|1.6987643617021269|
+------------+------------------+



In [36]:
# Extended aggregation: count, avg, min, max per payment type
df.groupBy('payment_type') \
  .agg(
      count('*').alias('trip_count'),
      avg('total_amount').alias('avg_amount'),
      spark_min('total_amount').alias('min_amount'),
      spark_max('total_amount').alias('max_amount')
  ) \
  .sort('payment_type') \
  .show()

+------------+----------+------------------+----------+----------+
|payment_type|trip_count|        avg_amount|min_amount|max_amount|
+------------+----------+------------------+----------+----------+
|           0|   1088058| 31.68379475177417|    -47.04|    684.19|
|           1|   2249747| 29.61055066192971|   -158.05|    2500.0|
|           2|    314043|23.411882226317573|    -827.7|    911.71|
|           3|     16641| 8.927757947238725|   -2560.2|    2560.2|
|           4|     56400|1.6987643617021269|   -904.25|    904.25|
+------------+----------+------------------+----------+----------+



In [37]:
# ── 13. WRAP-UP ───────────────────────────────────────────────────────────────

print("Analysis complete. Stopping Spark session.")
spark.stop()

Analysis complete. Stopping Spark session.
